# Chạy Benchmark LVBench trên Google Colab (Multi-Agent LensWalk)

Notebook này đóng gói toàn bộ quy trình chạy và đánh giá hệ thống **Orchestrator – Parallel Actors – Stitched Verify** (branch `feat/multi-agent` của `LensWalk`) trên tập dữ liệu **LVBench** (`https://github.com/zai-org/LVBench`).

**Các bước trong notebook:**
1. Kiểm tra GPU/FFmpeg & Mount Google Drive để lưu video + kết quả.
2. Lấy mã nguồn `LensWalk` (branch `feat/multi-agent`) và cài đặt thư viện.
3. Thiết lập `OPENAI_API_KEY`, `OPENAI_API_BASE` & chạy Smoke Test offline.
4. Tải metadata `LVBench` + tải thử video (hoặc trỏ tới thư mục video trên Drive).
5. Chuyển đổi `video_info.meta.jsonl` → `lvbench_lenswalk.jsonl`.
6. Chạy inference Multi-Agent (và tùy chọn chạy đối chứng `release` baseline).
7. Thống kê Accuracy theo từng hạng mục câu hỏi của LVBench & phân tích Frame Budget.

## 1. Mount Google Drive & Kiểm tra môi trường Colab

In [1]:
from google.colab import drive
drive.mount('/content/drive')

!ffmpeg -version | head -n 2
!python --version

Mounted at /content/drive
ffmpeg version 6.1.1-3ubuntu5 Copyright (c) 2000-2023 the FFmpeg developers
built with gcc 13 (Ubuntu 13.2.0-23ubuntu3)
Python 3.13.18


## 2. Lấy mã nguồn `LensWalk` (`feat/multi-agent`) & Cài đặt thư viện

Chọn **1 trong 2 cách**:
- **Cách A**: Bạn đã copy/upload thư mục `LensWalk` (đang ở branch `feat/multi-agent`) lên Google Drive (`/content/drive/MyDrive/LensWalk`).
- **Cách B**: Clone từ repo GitHub của bạn và `git checkout feat/multi-agent`.

In [2]:
import os
from pathlib import Path

# --- CẤU HÌNH ĐƯỜNG DẪN REPO ---
USE_DRIVE_REPO = False  # Đặt True nếu bạn copy repo lên Google Drive
DRIVE_REPO_PATH = "/content/drive/MyDrive/LensWalk"
GITHUB_REPO_URL = "https://github.com/nhatsonle/LensWalk.git"
BRANCH_NAME = "feat/multi-agent"

WORK_REPO = Path("/content/LensWalk")
if USE_DRIVE_REPO:
    assert Path(DRIVE_REPO_PATH).exists(), f"Không tìm thấy {DRIVE_REPO_PATH} trên Drive!"
    if not WORK_REPO.exists():
        !cp -r "{DRIVE_REPO_PATH}" /content/LensWalk
else:
    if not WORK_REPO.exists():
        !git clone "{GITHUB_REPO_URL}" /content/LensWalk

%cd /content/LensWalk
!git checkout {BRANCH_NAME}
!git status -sb

Cloning into '/content/LensWalk'...
remote: Enumerating objects: 77, done.
remote: Counting objects: 100% (52/52), done.
remote: Compressing objects: 100% (46/46), done.
remote: Total 77 (delta 6), reused 28 (delta 6), pack-reused 25 (from 1)
Receiving objects: 100% (77/77), 7.34 MiB | 16.12 MiB/s, done.
Resolving deltas: 100% (7/7), done.
/content/LensWalk
branch 'feat/multi-agent' set up to track 'origin/feat/multi-agent'.
Switched to a new branch 'feat/multi-agent'
## feat/multi-agent...origin/feat/multi-agent


In [3]:
# Cài đặt các thư viện cần thiết trên Colab
!pip install -q -U litellm openai pydantic pyyaml pillow numpy pandas decord opencv-python-headless "yt-dlp[default]" deno huggingface_hub
!apt-get -qq -y install ffmpeg

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.1.1 which is incompatible.
numba 0.64.0 requires numpy<2.5,>=1.22, but you have numpy 2.5.5 which is incompatible.
gradio 6.31.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.


## 3. Cấu hình API Key (`litellm`) & Chạy Smoke Test offline

In [5]:
import os
from getpass import getpass

# Thử lấy từ Colab Secrets trước, nếu chưa có thì nhập tay
try:
    from google.colab import userdata
    api_key = userdata.get("OPENAI_API_KEY") or ""
    api_base = userdata.get("OPENAI_API_BASE") or ""
except Exception:
    api_key = ""
    api_base = ""

if not api_key:
    api_key = getpass("Nhập OPENAI_API_KEY: ").strip()
if not api_base:
    api_base = input("Nhập OPENAI_API_BASE (vd https://hub.smilesoftware.org/v1): ").strip()

os.environ["OPENAI_API_KEY"] = api_key
os.environ["OPENAI_API_BASE"] = api_base
print("Đã thiết lập OPENAI_API_BASE =", os.environ["OPENAI_API_BASE"])

Đã thiết lập OPENAI_API_BASE = https://hub.smilesoftware.org/v1


In [6]:
%cd /content/LensWalk
!PYTHONPATH=. python -u tests/mock_multi_agent_smoke.py

/content/LensWalk
OverviewObserver: not adding extra timestamps to VLM calls.
Loaded tool: overview_observer -> OverviewObserver
DynamicFocusObserver: using timestamps for VLM calls.
Loaded tool: dynamic_focus -> DynamicFocusObserver
ScanObserver: using timestamps for VLM calls.
Loaded tool: scan_observer -> ScanObserver
StitchedProbeObserver: using timestamps for VLM calls.
Loaded tool: stitched_observer -> StitchedProbeObserver
Loaded tool: submit_report -> SubmitReportTool
Loaded tool: finish -> FinishTool
[PASS] Case 1 (Explicit timestamp -> conditional overview skip -> FINISH)
OverviewObserver: not adding extra timestamps to VLM calls.
Loaded tool: overview_observer -> OverviewObserver
DynamicFocusObserver: using timestamps for VLM calls.
Loaded tool: dynamic_focus -> DynamicFocusObserver
ScanObserver: using timestamps for VLM calls.
Loaded tool: scan_observer -> ScanObserver
StitchedProbeObserver: using timestamps for VLM calls.
Loaded tool: stitched_observer -> StitchedProbeObse

## 4. Tải dữ liệu LVBench (`video_info.meta.jsonl` & Video)

- Nếu bạn **đã tải sẵn** video LVBench lên Google Drive, chỉ cần sửa `LVBENCH_DIR` trỏ vào thư mục đó.
- Nếu chưa có, cell dưới đây sẽ clone repo `zai-org/LVBench`, tải `video_info.meta.jsonl` từ Hugging Face (`THUDM/LVBench`) và cho phép tải thử **N video đầu tiên** bằng `yt-dlp` để test nhanh.

In [9]:
!pip install -q -U "yt-dlp[default]" deno
!deno --version

deno 2.9.8 (stable, release, x86_64-unknown-linux-gnu)
v8 15.0.245.2-rusty
typescript 6.0.3


In [10]:
from pathlib import Path
import json
import subprocess

LVBENCH_DIR = Path("/content/drive/MyDrive/datasets/LVBench")
VIDEO_DIR = LVBENCH_DIR / "videos"
SUB_DIR = LVBENCH_DIR / "subtitles"  # tùy chọn nếu có phụ đề
META_PATH = LVBENCH_DIR / "video_info.meta.jsonl"

LVBENCH_DIR.mkdir(parents=True, exist_ok=True)
VIDEO_DIR.mkdir(parents=True, exist_ok=True)

# 1) Tải video_info.meta.jsonl nếu chưa có
if not META_PATH.exists():
    from huggingface_hub import hf_hub_download
    downloaded = hf_hub_download(
        repo_id="THUDM/LVBench",
        filename="video_info.meta.jsonl",
        repo_type="dataset",
        local_dir=str(LVBENCH_DIR),
    )
    print("Downloaded metadata to:", downloaded)
else:
    print("Metadata đã tồn tại:", META_PATH)

# 2) Tải thử N video bằng yt-dlp (đặt MAX_VIDEOS_TO_DOWNLOAD = 0 nếu bạn đã có sẵn video trên Drive)
MAX_VIDEOS_TO_DOWNLOAD = 5
COOKIES_FILE = "/content/drive/MyDrive/cookies.txt"  # Tùy chọn: "/content/drive/MyDrive/youtube_cookies.txt" nếu Colab bị YouTube chặn IP

if MAX_VIDEOS_TO_DOWNLOAD > 0:
    keys = []
    with META_PATH.open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                keys.append(json.loads(line)["key"])
    for vid_key in keys[:MAX_VIDEOS_TO_DOWNLOAD]:
        out_mp4 = VIDEO_DIR / f"{vid_key}.mp4"
        if out_mp4.exists():
            print(f"[Skip] {vid_key}.mp4 already exists")
            continue
        url = f"https://www.youtube.com/watch?v={vid_key}"
        print(f"[Download] {url} -> {out_mp4}")
        cmd = [
            "yt-dlp",
            "-f", "bv*[height<=480][ext=mp4]+ba[ext=m4a]/b[height<=480]/bv*[height<=480]+ba/b",
            "--merge-output-format", "mp4",
            "-o", str(out_mp4),
        ]
        if COOKIES_FILE and Path(COOKIES_FILE).exists():
            cmd.extend(["--cookies", COOKIES_FILE])
        cmd.append(url)
        proc = subprocess.run(cmd, capture_output=True, text=True, check=False)
        if proc.returncode != 0:
            err_tail = (proc.stderr or proc.stdout or "").strip().splitlines()[-5:]
            print(f"  [Error {vid_key}] code={proc.returncode}: " + " | ".join(err_tail))

print("Tổng số video .mp4 hiện có:", len(list(VIDEO_DIR.glob("*.mp4"))))

Metadata đã tồn tại: /content/drive/MyDrive/datasets/LVBench/video_info.meta.jsonl
[Download] https://www.youtube.com/watch?v=Cm73ma6Ibcs -> /content/drive/MyDrive/datasets/LVBench/videos/Cm73ma6Ibcs.mp4
[Download] https://www.youtube.com/watch?v=q01CUy_gwdU -> /content/drive/MyDrive/datasets/LVBench/videos/q01CUy_gwdU.mp4
[Download] https://www.youtube.com/watch?v=28CIeC8cZks -> /content/drive/MyDrive/datasets/LVBench/videos/28CIeC8cZks.mp4
  [Error 28CIeC8cZks] code=1: ERROR: [youtube] 28CIeC8cZks: Video unavailable. This video is no longer available because the YouTube account associated with this video has been terminated.
[Download] https://www.youtube.com/watch?v=TJR1oYDDTwg -> /content/drive/MyDrive/datasets/LVBench/videos/TJR1oYDDTwg.mp4
[Download] https://www.youtube.com/watch?v=HfEVEGf1A8Q -> /content/drive/MyDrive/datasets/LVBench/videos/HfEVEGf1A8Q.mp4
Tổng số video .mp4 hiện có: 4


## 5. Chuyển `video_info.meta.jsonl` → `lvbench_lenswalk.jsonl`

In [11]:
import json

FLAT_JSONL_PATH = LVBENCH_DIR / "lvbench_lenswalk.jsonl"

qa_count = 0
video_count = 0
with META_PATH.open("r", encoding="utf-8") as fin, FLAT_JSONL_PATH.open("w", encoding="utf-8") as fout:
    for line in fin:
        if not line.strip():
            continue
        item = json.loads(line)
        key = item["key"]
        video_file = VIDEO_DIR / f"{key}.mp4"
        if not video_file.exists():
            continue
        video_count += 1

        sub_file = None
        if SUB_DIR.exists():
            for ext in (".srt", ".json"):
                cand = SUB_DIR / f"{key}{ext}"
                if cand.exists():
                    sub_file = cand.name
                    break

        for qa in item.get("qa", []):
            row = {
                "id": f"{key}_{qa.get('uid', qa_count)}",
                "video_path": f"{key}.mp4",
                "question": qa["question"],
                "answer": qa["answer"],
                "question_type": "multiple-choice",
                "lvbench_type": qa.get("question_type", []),
                "video_category": item.get("type", ""),
                "time_reference": qa.get("time_reference", ""),
            }
            if sub_file:
                row["subtitle_path"] = sub_file
            fout.write(json.dumps(row, ensure_ascii=False) + "\n")
            qa_count += 1

print(f"Đã đóng gói {qa_count} câu hỏi từ {video_count} video -> {FLAT_JSONL_PATH}")

Đã đóng gói 66 câu hỏi từ 4 video -> /content/drive/MyDrive/datasets/LVBench/lvbench_lenswalk.jsonl


## 6. Chạy Benchmark Multi-Agent (lưu kết quả trực tiếp vào Google Drive)

In [12]:
%cd /content/LensWalk

OUTPUT_DIR = "/content/drive/MyDrive/LensWalk_Results"
MODEL_NAME = "gpt-6.1-terra"   # hoặc "gpt-5", "gpt-4o", ...
TOTAL_BUDGET = 192
OVERVIEW_FRAMES = 64
VERIFY_RESERVE = 32
DATA_START = 0
DATA_END = 20                  # Đặt -1 để chạy toàn bộ file JSONL
WORKERS = 1                    # Nên để 1 nếu API proxy bị giới hạn rate limit (429)
MAX_ACTOR_WORKERS = 1          # Tăng lên 3 nếu endpoint cho phép gọi song song mạnh
USE_SUBTITLE = False           # Đặt True nếu có thư mục SUB_DIR
RUN_TAG = "lvbench_colab"

!PYTHONPATH=. python -u eval/infer_bench_multi_agent.py \
  --tool_config tool_configs/vcs_multi_agent.yaml \
  --bench_name "{FLAT_JSONL_PATH}" \
  --video_root "{VIDEO_DIR}" \
  --subtitle_root "{SUB_DIR}" \
  --use_subtitle {USE_SUBTITLE} \
  --model_name_or_path "{MODEL_NAME}" \
  --total_frame_budget {TOTAL_BUDGET} \
  --overview_frames {OVERVIEW_FRAMES} \
  --verify_reserve_frames {VERIFY_RESERVE} \
  --max_actor_turns 3 \
  --max_replans 1 \
  --workers {WORKERS} \
  --max_actor_workers {MAX_ACTOR_WORKERS} \
  --data_start {DATA_START} \
  --data_end {DATA_END} \
  --output_dir "{OUTPUT_DIR}" \
  --custom_name "{RUN_TAG}" \
  --verbose

/content/LensWalk
OverviewObserver: not adding extra timestamps to VLM calls.
Loaded tool: overview_observer -> OverviewObserver
DynamicFocusObserver: using timestamps for VLM calls.
Loaded tool: dynamic_focus -> DynamicFocusObserver
ScanObserver: using timestamps for VLM calls.
Loaded tool: scan_observer -> ScanObserver
StitchedProbeObserver: using timestamps for VLM calls.
Loaded tool: stitched_observer -> StitchedProbeObserver
Loaded tool: submit_report -> SubmitReportTool
Loaded tool: finish -> FinishTool
[Session Cm73ma6Ibcs_56] Video duration=3665.5s | total_frame_budget=192
[Phase 0: Overview] Evaluating whether overview is needed...
[Session Cm73ma6Ibcs_55] Video duration=3665.5s | total_frame_budget=192
[Phase 0: Overview] Evaluating whether overview is needed...
[Phase 0: Overview] Completed | frames=0 | remaining_budget=192
[Phase 1: Orchestrator Plan] Generating hypotheses...
[Phase 0: Overview] Completed | frames=0 | remaining_budget=192
[Phase 1: Orchestrator Plan] Genera

In [14]:
import os, litellm
print("BASE:", os.getenv("OPENAI_API_BASE"), "| KEY_LEN:", len(os.getenv("OPENAI_API_KEY", "")))
resp = litellm.completion(
    model="gpt-6-luna",
    api_key=os.getenv("OPENAI_API_KEY"),
    api_base=os.getenv("OPENAI_API_BASE"),
    messages=[{"role": "user", "content": "ping"}],
)
print(resp.choices[0].message.content)

BASE: https://hub.smilesoftware.org/v1 | KEY_LEN: 25

Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm.turn_on_debug()'.



ServiceUnavailableError: litellm.ServiceUnavailableError: ServiceUnavailableError: OpenAIException - litellm.ServiceUnavailableError: ServiceUnavailableError: OpenAIException - [codex/gpt-6-luna] [429]: The usage limit has been reached (reset after 36s)

LiteLLM: model group 'gpt-6-luna' failed with the error above. No fallback was attempted.

### (Tùy chọn) Chạy đối chứng Baseline trên branch `release`
Bỏ comment nếu bạn muốn chạy song song bản LensWalk gốc trên cùng tập dữ liệu để so sánh.

In [ ]:
# !pip install -q "transformers>=4.45,<5" xlsxwriter
# %cd /content/LensWalk
# !git checkout release
# import os
# os.environ["OPENAI_BASE_URL"] = os.environ["OPENAI_API_BASE"]
# os.environ["TOOL_OPENAI_BASE_URL"] = os.environ["OPENAI_API_BASE"]
# os.environ["TOOL_OPENAI_API_KEY"] = os.environ["OPENAI_API_KEY"]
# os.environ["MLLM_BACKEND"] = MODEL_NAME
# !PYTHONPATH=. python -u eval/infer_bench_summ_vcs.py \
#   --tool_config tool_configs/vcs_standard.yaml \
#   --bench_name "{FLAT_JSONL_PATH}" \
#   --video_root "{VIDEO_DIR}" \
#   --model_name_or_path "{MODEL_NAME}" \
#   --max_turns 5 \
#   --workers {WORKERS} \
#   --data_start {DATA_START} \
#   --data_end {DATA_END} \
#   --custom_name "lvbench_release_baseline"

## 7. Thống kê Accuracy theo từng hạng mục LVBench & Phân tích Frame Budget

In [ ]:
import json
from pathlib import Path
from collections import defaultdict
import pandas as pd

runs_root = Path(OUTPUT_DIR) / "run_multi_agent"
run_dirs = sorted([d for d in runs_root.iterdir() if d.is_dir()], key=lambda p: p.stat().st_mtime)
assert run_dirs, f"Chưa có kết quả trong {runs_root}"
latest_run = run_dirs[-1]
print("Đang phân tích run mới nhất:", latest_run)

summary = json.loads((latest_run / "summary_metrics.json").read_text(encoding="utf-8"))
print("=== TỔNG QUAN ===")
print(json.dumps(summary, indent=2, ensure_ascii=False))

# Map id -> lvbench_type từ FLAT_JSONL_PATH
meta_by_id = {}
with FLAT_JSONL_PATH.open("r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            r = json.loads(line)
            meta_by_id[str(r["id"])] = r

cat_scores = defaultdict(list)
rows = []
with (latest_run / "all_conversations.jsonl").open("r", encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue
        rec = json.loads(line)
        qid = str(rec.get("id"))
        m = meta_by_id.get(qid, {})
        types = m.get("lvbench_type") or ["unknown"]
        if isinstance(types, str):
            types = [types]
        for t in types:
            cat_scores[t].append(float(rec.get("score", 0.0)))
        rows.append({
            "id": qid,
            "score": rec.get("score"),
            "pred": rec.get("answer"),
            "gt": rec.get("golden_answer"),
            "frames": rec["budget"]["total_frames_used"],
            "stitched": rec.get("has_stitched_verified"),
            "replan": rec.get("replan_count"),
            "elapsed_s": rec.get("elapsed_sec"),
        })

cat_df = pd.DataFrame([
    {"lvbench_type": k, "count": len(v), "accuracy": round(sum(v) / len(v) * 100, 2)}
    for k, v in sorted(cat_scores.items())
])
print("\n=== ACCURACY THEO NHÓM CÂU HỎI LVBENCH (%) ===")
display(cat_df)
display(pd.DataFrame(rows).head(20))